# Jumbotail Replenishment Planner

**Suggestion generation date: 2026-03-16**

This notebook implements the take-home requirements, parses the JSON fields, accounts for on-hand inventory and open POs, rounds to cases, respects the space cap, applies MOV, and produces the required output columns.

In [1]:
import pandas as pd, numpy as np, json, math
df = pd.read_csv('assignment_data.csv')
print('Dataset shape:', df.shape)
print('Columns:', len(df.columns))

Dataset shape: (1389, 36)
Columns: 36


## JSON parsing
`inventory_breakup` and `open_po_details` are parsed as JSON. The aggregate `orderedquantity` is used as the total open-PO pipeline because it is the explicit dataset-level total; the JSON detail is retained for audit/reconciliation.

In [2]:
def parse_json(s):
    if pd.isna(s) or not str(s).strip(): return {}
    try:
        x=json.loads(s)
        return x if isinstance(x,dict) else {}
    except Exception:
        return {}

def po_items(s): return list(parse_json(s).values())

df["_open_po_units"]=pd.to_numeric(df["orderedquantity"],errors="coerce").fillna(0)

print("Rows:",len(df))
print("Aggregate open-PO units:",int(df["_open_po_units"].sum()))

Rows: 1389
Aggregate open-PO units: 1133267


## Replenishment logic
Target DOI = `inv_norm + safety_stock`.

Required units = max(target units minus on-hand minus open-PO units, 0).

Required units are rounded up to whole cases. The order is raised toward the vendor's MOV and capped at the largest whole-case quantity that does not exceed `max_allocated_space`. If MOV still cannot be reached within that cap, the planner places the maximum feasible whole-case order and flags `MOV_NOT_MET_SPACE_LIMIT`.

In [3]:
from replenishment_planner import build_planner
result = build_planner(pd.read_csv('assignment_data.csv'))
print(result[['jpin','final_suggestion','final_cases_suggestion','final_value',
              'final_days_of_inventory','mov_check']].head(5).to_string())

        jpin  final_suggestion  final_cases_suggestion  final_value  final_days_of_inventory                mov_check
0  SKU-00786                 0                       0          0.0                      NaN             NOT_REQUIRED
1  SKU-00726                 0                       0          0.0                    23.67             NOT_REQUIRED
2  SKU-00846                 0                       0          0.0                    16.00             NOT_REQUIRED
3  SKU-00352               800                      20       5552.0                    15.25  MOV_NOT_MET_SPACE_LIMIT
4  SKU-00725                 0                       0          0.0                    19.34             NOT_REQUIRED


## Constraint validation

In [4]:
print("Whole-case:", (result.final_suggestion == result.final_cases_suggestion*result.case_size).all())
print("Space cap:", (result.final_suggestion <= result.max_allocated_space).all())
print("Non-negative:", (result.final_suggestion >= 0).all())
print("Suggested units:", int(result.final_suggestion.sum()))
print("Suggested cases:", int(result.final_cases_suggestion.sum()))
print("Suggested value:", round(result.final_value.sum(),2))
print("MOV exceptions (space-limited):", int((result.mov_check=="MOV_NOT_MET_SPACE_LIMIT").sum()))
print()
print(result.mov_check.value_counts())

Whole-case: True
Space cap: True
Non-negative: True
Suggested units: 1309732
Suggested cases: 14153
Suggested value: 13531485.1
MOV exceptions (space-limited): 703

mov_check
MOV_NOT_MET_SPACE_LIMIT    703
NOT_REQUIRED               669
PASS                        13
SPACE_LIMIT                   4
Name: count, dtype: int64


## SQL
See `replenishment_queries.sql` for the required vendor-level dashboard query, top-10 risk query, and a bonus sales-band inventory-health query.